In [ ]:
%run ./_bootstrap

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from helpers.retriever_train import train_tfidf_knn, default_retriever_params
from helpers.mlflow_retriever import log_retriever_artifacts

In [ ]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [ ]:
# load doc_chunks
chunks_df = spark.table("workspace.med.doc_chunks")
chunks_df = chunks_df.filter(F.col("chunk_text").isNotNull() & (F.length("chunk_text") > 0))

In [ ]:
# convert to pandas
chunks_pdf = chunks_df.select("chunk_id", "chunk_text").toPandas()

In [ ]:
chunk_ids = chunks_pdf["chunk_id"].tolist()
texts = chunks_pdf["chunk_text"].tolist()

In [ ]:
vectorizer, knn, matrix = train_tfidf_knn(texts, n_neighbors=10)

In [ ]:
params = default_retriever_params(n_neighbors=10)
metadata = {"source_table": "workspace.med.doc_chunks", "num_chunks": len(chunk_ids)}

In [ ]:

run_id = log_retriever_artifacts(vectorizer, knn, chunk_ids, metadata, params)
print("MLflow run_id:", run_id)

In [ ]:
(
  spark.createDataFrame([(run_id,)], ["train_run_id"])
       .withColumn("created_at", F.current_timestamp())
       .write.format("delta")
       .mode("append")
       .saveAsTable("workspace.med.retriever_registry")
)

print("Recorded latest train_run_id in workspace.med.retriever_registry:", run_id)